In [1]:
import pandas as pd,numpy as np,json
from sklearn.metrics.pairwise import cosine_distances
import getpass,os
from langchain.chat_models import init_chat_model
from ai_patterns_mining import parse_json_safe,Config
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from time import time
from langchain.tools import tool
from pydantic import BaseModel, Field
import seaborn as sns
import matplotlib.pyplot as plt
from langchain.agents.structured_output import ToolStrategy
from tqdm import tqdm

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pattern_descriptions=[
    {
    "Pattern Name": "Model Abstraction Pattern",
    "Problem": "Applications tightly coupled to a specific LLM provider (e.g., GPT, Gemini, Llama) become difficult to maintain, expensive to adapt, and vulnerable to vendor lock-in.",
    "Context": "Systems needing flexibility to switch between different LLMs based on cost, performance, latency, or domain requirements.",
    "Solution": "Introduce an abstraction layer that exposes a unified interface for interacting with any LLM. This layer handles prompt formatting, API calls, error handling, and response parsing, allowing the underlying model to be swapped without changing application logic.",
    "Result": "Developers gain the ability to seamlessly switch LLM providers, reduce vendor dependence, simplify maintenance, and ensure long-term system adaptability.",
    "Related Patterns": "Unified API Wrapper, Provider-Agnostic Architecture, LLM Routing Layer",
    "Category": "Architecture & Integration",
    "Uses": "Building scalable LLM applications, switching between providers dynamically, benchmarking models, reducing operational risk.",
    "Thinking": "This pattern abstracts away implementation details so the system remains stable even when the underlying LLM changes, ensuring long-term flexibility and maintainability."
  },
  {
    "Pattern Name": "Classical Models",
    "Problem": "Many tasks require models trained directly on specific datasets rather than relying solely on general-purpose pretrained LLMs. Without traditional ML or DL models, certain structured, numerical, or domain-specific problems cannot be solved effectively.",
    "Context": "Situations involving structured data, numerical prediction, image classification, or text processing tasks where the model must learn features directly from the provided dataset.",
    "Solution": "Use traditional machine learning and deep learning algorithms such as logistic regression, SVMs, decision trees, CNNs, and RNNs, which learn patterns from scratch or from domain-specific datasets.",
    "Result": "Provides interpretable, efficient, and domain-tailored models that perform well on structured data and scenarios where fine-tuning LLMs is unnecessary or overkill.",
    "Related Patterns": "Feature Engineering Pipeline, Text Vectorization Methods, Hybrid Modeling",
    "Category": "Machine Learning & Deep Learning Foundations",
    "Uses": "Classification, regression, clustering, computer vision tasks, numerical forecasting, and domain-specific supervised learning.",
    "Thinking": "Traditional ML and DL remain essential because many tasks demand custom patterns learned directly from data, making them complementary—not replacements—to modern LLM-based workflows."
  },
  {
    "Pattern Name": "Preprocessing Text and Numerical Data",
    "Problem": "Raw data often contains noise, inconsistent formats, missing values, and unstructured content. Without preprocessing, models learn incorrectly and produce poor results.",
    "Context": "Machine learning pipelines that require numerical stability, cleaned inputs, and feature transformations for both text and numerical datasets.",
    "Solution": "Apply preprocessing transformations: for numerical data, perform scaling, normalization, imputation, and feature encoding; for text, perform tokenization, stop-word removal, stemming/lemmatization, and convert text into numerical representations using TF-IDF, word embeddings, or other vectorizers.",
    "Result": "Improves model stability, accuracy, training efficiency, and ensures data is in a usable form for downstream ML models.",
    "Related Patterns": "Feature Engineering Pipeline, Vectorization Techniques, Data Cleaning Pattern",
    "Category": "Data Processing & Preparation",
    "Uses": "Cleaning raw datasets, preparing structured and unstructured data for ML, improving training quality, enabling classical ML algorithms to work effectively.",
    "Thinking": "Preprocessing is fundamental because machine learning models depend on well-structured, normalized inputs; without it, even the best algorithms perform poorly."
  }
]

In [3]:
# Build LLM helper to map None rows to one of the proposed new patterns
from langchain_core.prompts import ChatPromptTemplate
from typing import Optional
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")
    
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

In [4]:
yes_no_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strict binary classifier. Decide if the given code summary fits the pattern description. Be strict when classifying. say YES if it fully implements the mentioned pattern otherwise say no"),
    ("human", "Pattern description:\n{pattern}\n\nCode summary:\n{code_summary}\n\nAnswer with exactly YES or NO."),
])
def pattern_matches_code(code_summary: str, pattern: str) -> str:
    msgs = yes_no_prompt.format_messages(code_summary=code_summary, pattern=pattern)
    resp = llm.invoke(msgs)
    answer = resp.content.strip().upper()
    return "YES" if "YES" in answer else "NO"

In [5]:
def pattern_combiner(pattern):
    pattern_name = pattern['Pattern Name']
    Problem = pattern['Problem']
    Context = pattern['Context']
    Solution = pattern['Solution']
    Result = pattern['Result']
    Related_Patterns = pattern['Related Patterns']
    Category = pattern['Category']
    Uses = pattern['Uses']
    return f"""
Pattern Name: {pattern_name}
Problem: {Problem}
Context: {Context}
Solution: {Solution}
Result: {Result}
Related Patterns: {Related_Patterns}
Category: {Category}
Uses: {Uses}
"""

In [6]:
code_files = pd.read_json("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/pattern_classification_verification_results_NN_v2_list.json")
code_files = code_files['code file']

In [13]:
pattern_tree_path = "results/pattern_tree_classification_llms_for_new_patterns_v1.csv"
if os.path.exists(pattern_tree_path):
    pattern_tree = pd.read_csv(pattern_tree_path)
else:
    pattern_tree= pd.DataFrame(columns=['code_file','Model Abstraction Pattern','Classical Models','Preprocessing Text and Numerical Data'])
for file in tqdm(code_files):
    _file = file.replace('https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/','/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/repo_callgraph_clusters/')    
    if file in pattern_tree['code_file'].values:
        continue
    with open(_file,'r') as cf:
        code_summary = cf.read()
    pattern_tree.loc[len(pattern_tree)] = [
        file,
        pattern_matches_code(code_summary,pattern_combiner(pattern_descriptions[0])),
        pattern_matches_code(code_summary,pattern_combiner(pattern_descriptions[1])),
        pattern_matches_code(code_summary,pattern_combiner(pattern_descriptions[2]))
    ]
    pattern_tree.to_csv(pattern_tree_path,index=False)

  0%|          | 0/1541 [00:00<?, ?it/s]

 82%|████████▏ | 1263/1541 [00:19<00:00, 6372.76it/s]Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised DeadlineExceeded: 504 Deadline Exceeded.
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised DeadlineExceeded: 504 Deadline Exceeded.
100%|██████████| 1541/1541 [20:30<00:00,  1.25it/s]


In [16]:
pattern_tree

,code_file,Model Abstraction Pattern,Classical Models,Preprocessing Text and Numerical Data
0,https://github.com/HasinthakaPiyumal/AI-Patter...,NO,YES,NO
